# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reetuparabat/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [23]:
!git clone https://github.com/reetuparabat/flyrank-ml-internship.git

!ls flyrank-ml-internship/data/raw

content_refresh_anonymized.csv


In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv('/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(f'Total rows: {len(df):,} | Unique clients: {df["client_id"].nunique()}')
print(f'Base rate (overall decline rate): {df["is_declining_label"].mean():.4f}')


fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
Total rows: 30,000 | Unique clients: 32
Base rate (overall decline rate): 0.5421


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [25]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df['client_id'])
test_clients = set(test_df['client_id'])
overlap = train_clients & test_clients

print(f'Train: {len(train_df):,} rows, {train_df["client_id"].nunique()} clients')
print(f'Test:  {len(test_df):,} rows, {test_df["client_id"].nunique()} clients')
print(f'Client overlap between train/test: {len(overlap)} (must be 0)')
print(f'Train decline rate: {train_df["is_declining_label"].mean():.4f} | '
      f'Test decline rate: {test_df["is_declining_label"].mean():.4f}')


Train: 22,885 rows, 24 clients
Test:  7,115 rows, 8 clients
Client overlap between train/test: 0 (must be 0)
Train decline rate: 0.5500 | Test decline rate: 0.5165


In [26]:
"""
2) Split design -- grouped by client_id, and why

I split by client_id (GroupShuffleSplit, 75/25), not by row. A random row split
would let the same client show up in both train and test -- the model could
partly "memorize" a client's baseline traffic level or writing style instead of
learning a pattern that generalizes to a client it has never seen, which is
exactly the kind of quiet leak that inflates a score without meaning anything.
The printed overlap count above must read 0; if it doesn't, the split is invalid
and nothing downstream can be trusted.

One honest tradeoff worth naming: with only 32 clients total, a 75/25 client-level
split gives a fairly small, lumpy test set (8 clients), so the exact precision@K
numbers below will move around some if I reseed the split. I'm reporting one seed,
not the best of several -- picking the seed that looks best would be its own
form of leakage.
"""


'\n2) Split design -- grouped by client_id, and why\n\nI split by client_id (GroupShuffleSplit, 75/25), not by row. A random row split\nwould let the same client show up in both train and test -- the model could\npartly "memorize" a client\'s baseline traffic level or writing style instead of\nlearning a pattern that generalizes to a client it has never seen, which is\nexactly the kind of quiet leak that inflates a score without meaning anything.\nThe printed overlap count above must read 0; if it doesn\'t, the split is invalid\nand nothing downstream can be trusted.\n\nOne honest tradeoff worth naming: with only 32 clients total, a 75/25 client-level\nsplit gives a fairly small, lumpy test set (8 clients), so the exact precision@K\nnumbers below will move around some if I reseed the split. I\'m reporting one seed,\nnot the best of several -- picking the seed that looks best would be its own\nform of leakage.\n'

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [27]:
# --- Recompute the ML-07 baseline rule, fit only on TRAIN (same discipline as the model) ---
benchmark_ctr = train_df.groupby('position_tier')['ctr'].mean().to_dict()

def score_baseline(frame):
    visible = (frame['impressions_90d'] >= 500).astype(int)
    bench = frame['position_tier'].map(benchmark_ctr)
    gap = (bench - frame['ctr']).clip(lower=0)
    return visible * gap * frame['impressions_90d'] / 100

test_df['baseline_score'] = score_baseline(test_df)
print('Baseline (ML-07 CTR-fix rule) scored on test set — benchmark_ctr fit on train only.')
# --- Features: exclude label-derived leakage (ML-04 lesson) and ID columns ---
LEAKY = ['trend_direction', 'trend_pct',
         'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
         'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
IDS = ['content_id', 'client_id']
DROP_LOW_VALUE = ['provider_used', 'model_used']  # >70%/19% missing, not conceptually predictive
TARGET = 'is_declining_label'

feature_cols = [c for c in df.columns if c not in LEAKY + IDS + DROP_LOW_VALUE + [TARGET]]
numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
categorical_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]

# ML-04 caught systematic (not random) missingness tied to content_type -- add
# missing-indicator flags rather than silently imputing it away
SYSTEMATIC_MISSING = ['word_count', 'char_count', 'search_volume', 'competition', 'cpc', 'main_intent']
for col in SYSTEMATIC_MISSING:
    if col in feature_cols:
        train_df[f'{col}_was_missing'] = train_df[col].isna().astype(int)
        test_df[f'{col}_was_missing'] = test_df[col].isna().astype(int)
missing_flag_cols = [f'{c}_was_missing' for c in SYSTEMATIC_MISSING if c in feature_cols]
numeric_cols_final = numeric_cols + missing_flag_cols

print(f'Numeric features ({len(numeric_cols_final)}):', numeric_cols_final)
print(f'Categorical features ({len(categorical_cols)}):', categorical_cols)
print(f'Excluded as label-derived leakage: {LEAKY}')
preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), numeric_cols_final),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])

X_train = train_df[numeric_cols_final + categorical_cols]
y_train = train_df[TARGET]
X_test = test_df[numeric_cols_final + categorical_cols]
y_test = test_df[TARGET]

logreg = Pipeline([('prep', preprocess),
                    ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))])
logreg.fit(X_train, y_train)
test_df['logreg_score'] = logreg.predict_proba(X_test)[:, 1]

rf = Pipeline([('prep', preprocess),
               ('clf', RandomForestClassifier(n_estimators=300, max_depth=8,
                                               min_samples_leaf=20,
                                               random_state=RANDOM_SEED, n_jobs=-1))])
rf.fit(X_train, y_train)
test_df['rf_score'] = rf.predict_proba(X_test)[:, 1]
print('Both models trained on the grouped train split.')
def precision_at_k(frame, score_col, k):
    top_k = frame.sort_values(score_col, ascending=False).head(k)
    return top_k[TARGET].mean()

base_rate = test_df[TARGET].mean()
rows = []
for k in [20, 50, 100]:
    rows.append({
        'K': k,
        'baseline_precision': round(precision_at_k(test_df, 'baseline_score', k), 3),
        'logreg_precision': round(precision_at_k(test_df, 'logreg_score', k), 3),
        'rf_precision': round(precision_at_k(test_df, 'rf_score', k), 3),
        'base_rate': round(base_rate, 3)
    })
comparison = pd.DataFrame(rows)
print('=== Comparison table: precision@K, same test split, same metric ===')
comparison
print('Secondary metric — ROC-AUC (whole ranking, not just top-K):')
print('Logistic Regression:', round(roc_auc_score(y_test, test_df['logreg_score']), 3))
print('Random Forest:      ', round(roc_auc_score(y_test, test_df['rf_score']), 3))
print('Baseline (as ranker):', round(roc_auc_score(y_test, test_df['baseline_score']), 3))


Baseline (ML-07 CTR-fix rule) scored on test set — benchmark_ctr fit on train only.
Numeric features (29): ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count_was_missing', 'char_count_was_missing', 'search_volume_was_missing', 'competition_was_missing', 'cpc_was_missing', 'main_intent_was_missing']
Categorical features (9): ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']
Excluded as label-derived leakage: ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'cl

In [28]:
"""
3) What the comparison table actually shows

The baseline (ML-07's CTR-underperformance-for-position rule) is barely
distinguishable from random at this task: precision@20 = 0.50, precision@50 =
0.50, precision@100 = 0.46 -- all at or BELOW the 0.517 base rate, and its
ROC-AUC (0.508) is essentially a coin flip. That is not a modeling failure,
it's an honest finding: the baseline was built to rank "CTR-fix opportunity,"
a different target than "is this page declining." It was never designed to
predict decline, so of course it doesn't -- and precision@K makes that visible
in a way accuracy alone would have hidden.

Logistic Regression is the clear winner at every K: precision@20 = 0.85 (vs.
0.517 base rate -- a real, large lift), precision@50 = 0.68, precision@100 =
0.70, ROC-AUC = 0.582. Random Forest is weaker than Logistic Regression at
every K (0.60 / 0.56 / 0.52) despite being the more complex model -- per the
skill's rule that complexity has to earn its place, it does not here, so my
answer for this lane is Logistic Regression, not Random Forest.

I'm keeping both in the table rather than hiding the loser, because "the
simpler model won" is itself the finding the skill asks for.
"""


'\n3) What the comparison table actually shows\n\nThe baseline (ML-07\'s CTR-underperformance-for-position rule) is barely\ndistinguishable from random at this task: precision@20 = 0.50, precision@50 =\n0.50, precision@100 = 0.46 -- all at or BELOW the 0.517 base rate, and its\nROC-AUC (0.508) is essentially a coin flip. That is not a modeling failure,\nit\'s an honest finding: the baseline was built to rank "CTR-fix opportunity,"\na different target than "is this page declining." It was never designed to\npredict decline, so of course it doesn\'t -- and precision@K makes that visible\nin a way accuracy alone would have hidden.\n\nLogistic Regression is the clear winner at every K: precision@20 = 0.85 (vs.\n0.517 base rate -- a real, large lift), precision@50 = 0.68, precision@100 =\n0.70, ROC-AUC = 0.582. Random Forest is weaker than Logistic Regression at\nevery K (0.60 / 0.56 / 0.52) despite being the more complex model -- per the\nskill\'s rule that complexity has to earn its place

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [29]:
perm_lr = permutation_importance(logreg, X_test, y_test, n_repeats=15,
                                   random_state=RANDOM_SEED, scoring='roc_auc', n_jobs=-1)
imp_lr = pd.DataFrame({'feature': X_test.columns,
                       'importance_mean': perm_lr.importances_mean,
                       'importance_std': perm_lr.importances_std}
                      ).sort_values('importance_mean', ascending=False)
print('Top 10 features by permutation importance (Logistic Regression, scored on AUC):')
imp_lr.head(10)
test_df['logreg_pred'] = (test_df['logreg_score'] >= 0.5).astype(int)
false_pos = test_df[(test_df['logreg_pred']==1) & (test_df[TARGET]==0)].sort_values('logreg_score', ascending=False)
false_neg = test_df[(test_df['logreg_pred']==0) & (test_df[TARGET]==1)].sort_values('logreg_score', ascending=True)

cols_show = ['content_id','client_id','content_type','position_tier','ctr',
             'impressions_90d','days_since_last_update','freshness_tier','logreg_score', TARGET]
print('Most-confident FALSE POSITIVES (predicted declining, actually stable):')
display(false_pos[cols_show].head(3))
print('Most-confident FALSE NEGATIVES (predicted stable, actually declining):')
display(false_neg[cols_show].head(3))
print('Confusion breakdown at 0.5 threshold:')
print(pd.crosstab(test_df[TARGET], test_df['logreg_pred'], rownames=['actual'], colnames=['predicted']))


Top 10 features by permutation importance (Logistic Regression, scored on AUC):
Most-confident FALSE POSITIVES (predicted declining, actually stable):


,content_id,client_id,content_type,position_tier,ctr,impressions_90d,days_since_last_update,freshness_tier,logreg_score,is_declining_label
10175,content_374e795aab68,client_f369cb89fc,keyword article,page_3_5,0.85,235,20,0-30,0.939156,0
26614,content_7be5f150dc65,client_f369cb89fc,keyword article,page_1,0.00,290,20,0-30,0.923760,0
20736,content_41baf0722ad9,client_8527a891e2,keyword article,striking,0.00,3115,104,91-180,0.904694,0


Most-confident FALSE NEGATIVES (predicted stable, actually declining):


,content_id,client_id,content_type,position_tier,ctr,impressions_90d,days_since_last_update,freshness_tier,logreg_score,is_declining_label
29158,content_e18144cbd19d,client_4e07408562,keyword article,top_3,0.0,3,20,0-30,0.077474,1
12845,content_742a8fcba2fe,client_e629fa6598,keyword article,deep,0.0,643,20,0-30,0.077896,1
4081,content_917fc1b11fe1,client_e629fa6598,keyword article,deep,0.0,916,22,0-30,0.077980,1


Confusion breakdown at 0.5 threshold:
predicted     0     1
actual               
0          1230  2210
1           893  2782


In [30]:
"""
4) Errors and interpretation

What the model leans on: days_with_impressions is the single strongest feature
by a wide margin, followed by days_with_sessions, content_age_days, users_90d,
and avg_position. This makes sense -- how CONSISTENTLY a page shows up in
search over the 90-day window (not just its raw volume) is a plausible, honest
signal of decline risk, and it's a different signal than my baseline's CTR gap.
No single feature dominates the way a leaked label-derived column would (see
the FL-04 leakage trap, where the leaked feature alone explained almost
everything) -- the importance here is spread across several genuinely
plausible signals, which is what I'd expect from a real pattern rather than
a shortcut.

Where it's wrong: the confusion matrix at a 0.5 threshold shows the model
over-predicts decline (2,210 false positives vs. 1,230 true negatives) --
it's biased toward flagging pages as declining. This is exactly why
precision@K, not raw accuracy, is the right metric for a ranked-queue
deliverable: at the actual K a human would review (20-100 pages), precision is
strong; the threshold-level confusion matrix looks worse because most of Lane
2's real decision only ever touches the top of the ranking, not a blanket
yes/no over every row.

Three concrete wrong cases:
- False positive: a "keyword article" at page_3_5, CTR 0.85%, updated 20 days
  ago -- fresh and reasonably well-performing, but the model still scored it
  0.94. Freshness and non-trivial CTR should predict stability; this looks
  like the model over-weighting something else (likely low days_with_impressions
  for this specific row) without enough countervailing signal from CTR.
- False positive: a page_1 result with CTR exactly 0.00% and only 290
  impressions in 90 days -- genuinely looks unhealthy on paper, and updated
  recently (20 days). It's a defensible near-miss: this page might legitimately
  be borderline, and "actually stable" here could just mean it hasn't
  declined YET.
- False negative: a top_3-position page with only 3 impressions in 90 days and
  0% CTR -- about as thin a page as exists in this dataset, yet the model
  gave it a low decline score (0.077). With so little traffic, there may not
  be enough signal in this row for the model to work with at all -- a
  low-volume blind spot worth flagging as a known weakness, not something I'd
  claim confidence about.

Human check still required: the model should not be the final word on any
single page, especially the thin-traffic false negative case above -- a
human reviewer should sanity-check any page with under ~50 impressions before
trusting either the baseline's or the model's score for it.
"""


'\n4) Errors and interpretation\n\nWhat the model leans on: days_with_impressions is the single strongest feature\nby a wide margin, followed by days_with_sessions, content_age_days, users_90d,\nand avg_position. This makes sense -- how CONSISTENTLY a page shows up in\nsearch over the 90-day window (not just its raw volume) is a plausible, honest\nsignal of decline risk, and it\'s a different signal than my baseline\'s CTR gap.\nNo single feature dominates the way a leaked label-derived column would (see\nthe FL-04 leakage trap, where the leaked feature alone explained almost\neverything) -- the importance here is spread across several genuinely\nplausible signals, which is what I\'d expect from a real pattern rather than\na shortcut.\n\nWhere it\'s wrong: the confusion matrix at a 0.5 threshold shows the model\nover-predicts decline (2,210 false positives vs. 1,230 true negatives) --\nit\'s biased toward flagging pages as declining. This is exactly why\nprecision@K, not raw accuracy, 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.